# Team Workbook — Day 2
## Descriptive & Diagnostic Analytics  ·  Unit II

Yesterday you described your data. Today you make it trustworthy, then make it explain itself.

**Two questions, all day:**

| | |
|---|---|
| **Descriptive** | *What happened?* — the numbers that summarise your data honestly |
| **Diagnostic** | *Why did it happen?* — the pattern underneath, and whether you believe it |

Same rule as yesterday: fill in the names in Step 0 and every cell below runs unchanged.

---


## Step 0 — Setup

Same four names as yesterday, plus one optional extra.


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 4)

FILE     = "yourfile.csv"
NUM_COL  = "____"          # your main measure
CAT_COL  = "____"          # your main category
DATE_COL = None            # or "your_date_column"
CAT2     = None            # a SECOND category, if you have one - used for drill-downs

df_raw = pd.read_csv(FILE)
df_raw.columns = df_raw.columns.str.strip()
df = df_raw.copy()

# --- make sure the measure really is a number (Day 1, Step 4) -----------------
before_nulls = df[NUM_COL].isnull().sum()
df[NUM_COL] = pd.to_numeric(df[NUM_COL], errors="coerce")
if df[NUM_COL].isnull().mean() > 0.3:
    df[NUM_COL] = pd.to_numeric(
        df_raw[NUM_COL].astype(str).str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce")
    print(f"NOTE: {NUM_COL} was text with symbols in it - stripped and converted.")
print(f"{NUM_COL}: {df[NUM_COL].notna().sum()} usable values of {len(df)}")

# --- quartile bands that survive ties and low-cardinality columns -------------
def make_bands(s, k=4):
    names = {2: ["low", "high"], 3: ["low", "mid", "high"],
             4: ["low", "mid-low", "mid-high", "high"]}
    for kk in range(k, 1, -1):
        try:
            b = pd.qcut(s, kk, duplicates="drop")
        except (ValueError, IndexError):
            continue
        n = b.cat.categories.size
        if n >= 2:
            return b.cat.rename_categories(names.get(n, [f"band{i+1}" for i in range(n)]))
    return None

print(df.shape)
df.head()


---
# Part A — Cleaning that changes the answer

Yesterday you counted the problems. Today you fix them — and you measure how much the fix moved your answer. That measurement is the point.


### A1 · The missingness map

Where are the holes, and are they random?


In [ ]:
miss = pd.DataFrame({
    "missing": df.isnull().sum(),
    "percent": (df.isnull().mean() * 100).round(1),
})
print(miss[miss["missing"] > 0].sort_values("missing", ascending=False))

plt.figure(figsize=(8, 3.2))
sns.heatmap(df.isnull(), cbar=False, cmap="Blues")
plt.title("Where the holes are (white = present, blue = missing)")
plt.tight_layout(); plt.show()


**Read the picture.** Horizontal stripes mean whole rows are broken. Vertical blocks mean one
column was never collected. Scattered dots mean random loss. Those three need different fixes.


### A2 · Is the missingness *related* to something?

This is the question almost everyone skips. If rows with a missing value are systematically different, deleting them biases your answer.


In [ ]:
hole_col = miss[miss["missing"] > 0].index[0] if (miss["missing"] > 0).any() else None

if hole_col and hole_col != NUM_COL:
    flag = df[hole_col].isnull()
    print(f"Comparing rows where {hole_col} is missing vs present:\n")
    print(df.groupby(flag)[NUM_COL].agg(["count", "mean", "median"]).round(2)
            .rename(index={False: f"{hole_col} present", True: f"{hole_col} MISSING"}))
    print(f"\n{CAT_COL} mix in each group (%):")
    print(pd.crosstab(flag, df[CAT_COL], normalize="index").mul(100).round(1))
else:
    print("No usable second column with holes - skip to A3.")


If the two rows of that table look different, your missing values are **not random**. Say so in your
report, and prefer filling over dropping. If they look the same, dropping is defensible.


### A3 · Four strategies, four different answers


In [ ]:
base = pd.to_numeric(df[NUM_COL], errors="coerce")

strategies = {
    "do nothing (skip nulls)": base,
    "drop the rows":           base.dropna(),
    "fill with the mean":      base.fillna(base.mean()),
    "fill with the median":    base.fillna(base.median()),
    "fill with GROUP mean":    base.fillna(df.groupby(CAT_COL)[NUM_COL].transform("mean")),
}
out = pd.DataFrame({
    k: {"rows": v.notna().sum(), "mean": round(v.mean(), 2),
        "median": round(v.median(), 2), "std": round(v.std(), 2)}
    for k, v in strategies.items()
}).T
out


**Look at how much the mean moves between rows of that table.** That spread is the size of the
decision you are making. Pick one strategy, write down which and why, and use it consistently.

> Filling with the group mean is usually the most honest of the four — it preserves the differences
> between your categories instead of flattening everyone toward one number.


In [ ]:
# apply your choice - edit this line
df[NUM_COL] = pd.to_numeric(df[NUM_COL], errors="coerce")
df[NUM_COL] = df[NUM_COL].fillna(df.groupby(CAT_COL)[NUM_COL].transform("median"))
print("Remaining nulls in", NUM_COL, ":", int(df[NUM_COL].isnull().sum()))


### A4 · Duplicates, types and category hygiene


In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before-len(df)} duplicate rows")

df[CAT_COL] = df[CAT_COL].astype(str).str.strip().str.title()
print(f"\n{CAT_COL} now has {df[CAT_COL].nunique()} labels:")
print(df[CAT_COL].value_counts())

if DATE_COL:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", format="mixed")
    print(f"\n{DATE_COL} parsed: {df[DATE_COL].notna().sum()} of {len(df)} rows, "
          f"{df[DATE_COL].min()} to {df[DATE_COL].max()}")


### A5 · Outliers — find them, then decide


In [ ]:
s  = df[NUM_COL]
q1, q3 = s.quantile(.25), s.quantile(.75)
iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
z  = (s - s.mean()) / s.std()

out_iqr = df[(s < lo) | (s > hi)]
out_z   = df[z.abs() > 3]
print(f"IQR rule  : {len(out_iqr)} outliers outside [{lo:,.2f}, {hi:,.2f}]")
print(f"z > 3 rule: {len(out_z)} outliers")

if len(out_iqr):
    cols = [c for c in [CAT_COL, NUM_COL] if c in df.columns]
    print("\nThe five most extreme:")
    print(out_iqr.reindex(out_iqr[NUM_COL].abs().sort_values(ascending=False).index)[cols].head().to_string(index=False))
    print(f"\nWhich {CAT_COL} do the outliers belong to?")
    print(out_iqr[CAT_COL].value_counts())


**Look at them before you touch them.** Three outcomes, and you must choose one and say which:

1. **Data-entry errors** (a price of 9,999,999, an age of 0) — remove them, and report how many.
2. **Real but extreme** (one genuinely huge order) — keep them, and quote the median as well as the mean.
3. **A whole category** (all the outliers are one group) — that is not an outlier problem, that is your
   finding. Investigate it in Part D.

> Never delete outliers silently. "We removed 14 rows above ₹50,000 because they were bulk orders,
> not retail" is good practice. Deleting them without saying so is not.


---
# Part B — Descriptive: what happened

The numbers that summarise your data honestly.


### B1 · Every numeric column at once


In [ ]:
num = df.select_dtypes(include=np.number)
desc = num.describe().T
desc["median"] = num.median()
desc["IQR"]    = num.quantile(.75) - num.quantile(.25)
desc["skew"]   = num.skew()
desc[["count","mean","median","std","min","max","IQR","skew"]].round(2)


**Three things to read off that table.**

- `mean` far from `median` → the distribution is lopsided; quote the median.
- `skew` beyond about ±1 → strongly lopsided. Positive means a long tail of large values.
- `std` larger than `mean` → enormous spread. An average is close to meaningless here; say so.


### B2 · Every category at once


In [ ]:
cats = [c for c in df.columns
        if df[c].dtype == object and 2 <= df[c].nunique() <= 25]
for c in cats[:5]:
    vc = df[c].value_counts()
    pct = (vc / vc.sum() * 100).round(1)
    print(f"--- {c}  ({df[c].nunique()} levels) ---")
    print(pd.DataFrame({"count": vc, "%": pct}).head(8).to_string())
    if pct.max() > 90:
        print("   NOTE: one level is over 90% of the data - this column cannot explain much.")
    print()


### B3 · The summary table — your main descriptive deliverable


In [ ]:
summary = (df.groupby(CAT_COL)[NUM_COL]
             .agg(n="count", mean="mean", median="median", std="std", min="min", max="max")
             .round(2)
             .sort_values("mean", ascending=False))
summary["share_of_rows_%"] = (summary["n"] / summary["n"].sum() * 100).round(1)
summary


**The `n` column governs everything else in that row.** A group of four can top the table by luck.
A useful habit: ignore any row where `n` is under about 20, and say in your report that you did.


### B4 · Distribution shape


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
sns.histplot(df[NUM_COL].dropna(), kde=True, ax=ax[0], color="#2194D4")
ax[0].axvline(df[NUM_COL].mean(),   color="#B4342F", ls="--", label="mean")
ax[0].axvline(df[NUM_COL].median(), color="#137A5E", ls="-",  label="median")
ax[0].legend(); ax[0].set_title(f"Distribution of {NUM_COL}")

sns.boxplot(x=df[NUM_COL].dropna(), ax=ax[1], color="#D8EAF7")
ax[1].set_title("The same data as a box plot")
plt.tight_layout(); plt.show()

print(f"skew {df[NUM_COL].skew():.2f}   |   mean {df[NUM_COL].mean():,.2f}   median {df[NUM_COL].median():,.2f}")


### B5 · Trend over time  *(only if you have a date)*


In [ ]:
if DATE_COL and df[DATE_COL].notna().sum() > 10:
    t = (df.dropna(subset=[DATE_COL])
           .set_index(DATE_COL)[NUM_COL]
           .resample("ME").agg(["count", "mean"]))
    print(t.round(2).head(12))
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(t.index, t["mean"], marker="o", color="#2194D4")
    ax.set_title(f"Monthly average {NUM_COL}"); ax.set_ylabel(NUM_COL)
    plt.tight_layout(); plt.show()
else:
    print("No date column - skip this step. Note it as a limitation in your report.")


---
# Part C — Six charts that earn their place

Not a gallery. Each of these answers a question you cannot answer from a table.


In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(14, 7))
P = "#2194D4"

sns.histplot(df[NUM_COL].dropna(), ax=ax[0,0], color=P, kde=True)
ax[0,0].set_title(f"1. Shape of {NUM_COL}")

order = list(df.groupby(CAT_COL)[NUM_COL].median().sort_values(ascending=False).index)
plot_df = df.dropna(subset=[NUM_COL, CAT_COL])
order = [o for o in order if o in set(plot_df[CAT_COL])]
sns.boxplot(data=plot_df, x=CAT_COL, y=NUM_COL, ax=ax[0,1], order=order, color="#D8EAF7")
ax[0,1].set_title(f"2. Spread within each {CAT_COL}"); ax[0,1].tick_params(axis="x", rotation=45)

g = df.groupby(CAT_COL)[NUM_COL].mean().sort_values(ascending=False)
sns.barplot(x=g.values, y=g.index, ax=ax[0,2], color=P)
ax[0,2].set_title(f"3. Average {NUM_COL} by {CAT_COL}")

nums = df.select_dtypes(include=np.number).columns.tolist()
other = [c for c in nums if c != NUM_COL]
if other:
    sns.regplot(data=df, x=other[0], y=NUM_COL, ax=ax[1,0],
                scatter_kws=dict(s=14, alpha=.5, color=P), line_kws=dict(color="#B4342F"))
    ax[1,0].set_title(f"4. {NUM_COL} vs {other[0]}")
else:
    ax[1,0].text(.5,.5,"only one numeric column",ha="center"); ax[1,0].set_axis_off()

if len(nums) > 1:
    sns.heatmap(df[nums].corr(numeric_only=True), annot=True, fmt=".2f", cmap="Blues",
                ax=ax[1,1], cbar=False, annot_kws={"size":7})
    ax[1,1].set_title("5. What moves with what")
else:
    ax[1,1].set_axis_off()

sns.countplot(data=df, y=CAT_COL, ax=ax[1,2],
              order=df[CAT_COL].value_counts().index, color=P)
ax[1,2].set_title(f"6. How many rows per {CAT_COL}")

plt.tight_layout(); plt.show()


**Chart 6 is the one people skip and the one that saves them.** It is the visual version of the
`n` column: it shows instantly whether chart 3's winner is built on four rows or four hundred.


---
# Part D — Diagnostic: why did it happen

Descriptive stops at *what*. Everything below is an attempt at *why* — and a check on whether you
should believe it.


### D1 · Group comparison with an effect size

Not just *is there a gap* — **how big**, and **on how many rows**.


In [ ]:
g = df.groupby(CAT_COL)[NUM_COL].agg(["count", "mean"]).sort_values("mean", ascending=False)
top, bottom = g.index[0], g.index[-1]
gap = g.loc[top, "mean"] - g.loc[bottom, "mean"]

print(f"Highest : {top:<20} {g.loc[top,'mean']:>10,.2f}   (n={int(g.loc[top,'count'])})")
print(f"Lowest  : {bottom:<20} {g.loc[bottom,'mean']:>10,.2f}   (n={int(g.loc[bottom,'count'])})")
print(f"Gap     : {gap:,.2f}   =  {gap / df[NUM_COL].mean() * 100:.1f}% of the overall average")
print(f"          in units of one standard deviation: {gap / df[NUM_COL].std():.2f} SD")


**The SD figure is the honest one.** A gap under about 0.2 SD is small however impressive the
percentage looks. Around 0.5 SD is a moderate difference. Above 0.8 SD is large.


### D2 · Correlation — and the two kinds


In [ ]:
nums = df.select_dtypes(include=np.number)
if nums.shape[1] > 1:
    pear = nums.corr(method="pearson")[NUM_COL].drop(NUM_COL).sort_values(key=abs, ascending=False)
    spear= nums.corr(method="spearman")[NUM_COL].drop(NUM_COL).sort_values(key=abs, ascending=False)
    print(f"What moves with {NUM_COL}:\n")
    print(pd.DataFrame({"Pearson (straight line)": pear.round(3),
                        "Spearman (any rising/falling)": spear.round(3)}).to_string())
    print("\nA big gap between the two columns means the relationship is real but curved.")
else:
    print("Only one numeric column - correlation needs at least two.")


Rules of thumb for the absolute value: below 0.3 weak, 0.3–0.6 moderate, above 0.6 strong.

**And the sentence you must never write:** *"X causes Y."* Correlation is consistent with X causing Y,
Y causing X, or something else causing both. You cannot tell which from this table.


### D3 · Cross-tab — two categories at once


In [ ]:
if CAT2:
    print("Counts:")
    print(pd.crosstab(df[CAT_COL], df[CAT2]))
    print("\nRow percentages - this is the one that shows the pattern:")
    print(pd.crosstab(df[CAT_COL], df[CAT2], normalize="index").mul(100).round(1))
    print(f"\nAverage {NUM_COL} in each cell:")
    print(pd.pivot_table(df, index=CAT_COL, columns=CAT2, values=NUM_COL,
                         aggfunc="mean").round(2))
else:
    print("Set CAT2 in Step 0 to a second category column to run this.")


### D4 · The drill-down — where the average hides the story


In [ ]:
if CAT2:
    drill = (df.groupby([CAT_COL, CAT2])[NUM_COL]
               .agg(n="count", mean="mean").round(2))
    drill = drill[drill["n"] >= 5].sort_values("mean", ascending=False)
    print("Every combination with at least 5 rows, best first:\n")
    print(drill.head(12).to_string())
    print("\n... and the weakest:\n")
    print(drill.tail(5).to_string())
else:
    print("Needs CAT2.")


### D5 · The reversal check (Simpson's paradox)

The most important cell in this workbook. A pattern that holds overall can **reverse** inside every
subgroup. If that happens to you and you do not check, your headline conclusion is backwards.


In [ ]:
if CAT2:
    overall = df.groupby(CAT_COL)[NUM_COL].mean().sort_values(ascending=False)
    print("OVERALL ranking:"); print(overall.round(2).to_string())

    print(f"\nRanking WITHIN each {CAT2}:")
    flipped = False
    for lvl, part in df.groupby(CAT2):
        if len(part) < 20: continue
        r = part.groupby(CAT_COL)[NUM_COL].mean().sort_values(ascending=False)
        print(f"\n  {CAT2} = {lvl}  (n={len(part)})")
        print("   " + r.round(2).to_string().replace("\n", "\n   "))
        common = [i for i in overall.index if i in r.index]
        if len(common) > 1 and list(r.index[:len(common)]) != [i for i in overall.index if i in r.index]:
            flipped = True

    print("\n" + ("!!! The order CHANGES inside subgroups - investigate before you conclude anything."
                  if flipped else
                  "The order holds inside subgroups. Your headline is safe."))
else:
    print("Needs CAT2. If you have no second category, say in your report that you could not run this check.")


### D6 · Write the diagnosis

Fill this in. It is the whole output of Part D.

- **The pattern:** ____________ has the highest/lowest ____________, by ______ (____ SD), on ____ rows.
- **What moves with it:** the strongest correlate is ____________ at r = ______.
- **Does it survive a drill-down?** yes / no / could not check, because ____________
- **Three explanations that would produce this pattern:**
  1. ____________
  2. ____________
  3. ____________
- **Which one the data cannot rule out:** ____________
- **What you would need to tell them apart:** ____________

---


# Part E — Feature engineering (the bridge to tomorrow)

Tomorrow you predict. A model can only use columns that exist, so today you make better ones.


In [ ]:
fe = df.copy()

# 1. a ratio or share - almost always more informative than the raw number
fe["share_of_group_%"] = (fe[NUM_COL] / fe.groupby(CAT_COL)[NUM_COL].transform("sum") * 100).round(2)

# 2. a distance-from-normal feature
fe["vs_group_mean"] = (fe[NUM_COL] - fe.groupby(CAT_COL)[NUM_COL].transform("mean")).round(2)

# 3. bands - turning a number into a category
bands = make_bands(fe[NUM_COL], 4)
if bands is None:
    print(f"NOTE: {NUM_COL} has too few distinct values to band - skipping the band column.")
else:
    fe[NUM_COL + "_band"] = bands
    print(f"Banded {NUM_COL} into {bands.cat.categories.size}: {list(bands.cat.categories)}")

# 4. a flag
fe["above_average"] = (fe[NUM_COL] >= fe[NUM_COL].mean()).astype(int)

# 5. date parts, if you have a date
if DATE_COL and fe[DATE_COL].notna().any():
    fe["month"]   = fe[DATE_COL].dt.month
    fe["weekday"] = fe[DATE_COL].dt.day_name()
    fe["is_weekend"] = fe[DATE_COL].dt.dayofweek.isin([5, 6]).astype(int)

print(fe.filter(regex="share_of|vs_group|_band|above_average|month|weekday|is_weekend").head())
if NUM_COL + "_band" in fe.columns:
    print("\nDoes the new band column separate the data?")
    print(fe.groupby(NUM_COL + "_band", observed=True)[NUM_COL].agg(["count", "mean"]).round(2))


### E2 · Encoding — turning categories into numbers


In [ ]:
# one-hot: for categories with no natural order (city, type, brand)
onehot = pd.get_dummies(fe[[CAT_COL]], columns=[CAT_COL], drop_first=True, dtype=int)
print("One-hot produced", onehot.shape[1], "columns:", list(onehot.columns)[:6])

# ordinal: ONLY where the order is real (low < mid < high)
if NUM_COL + "_band" in fe.columns:
    levels = list(fe[NUM_COL + "_band"].cat.categories)
    order_map = {lvl: i for i, lvl in enumerate(levels)}
    fe["band_code"] = fe[NUM_COL + "_band"].map(order_map)
    print("\nOrdinal coding of the band column:")
    print(fe[[NUM_COL + "_band", "band_code"]].drop_duplicates()
            .sort_values("band_code").to_string(index=False))
else:
    print("\nNo band column to encode.")

fe.to_csv("day2_clean.csv", index=False)
print("\nSaved day2_clean.csv - load THIS file tomorrow, not the raw one.")


> **Never ordinal-encode an unordered category.** Coding cities as 1, 2, 3 tells the model that
> city 3 is somehow three times city 1, and it will believe you.


---
## Day 2 deliverable

Add to your notebook, in markdown:

1. **The cleaning log** — what you dropped, what you filled, which strategy you chose and why, and how much the mean moved.
2. **The summary table** from B3, with the `n` column visible.
3. **Two charts** from Part C that actually support your finding — not all six.
4. **The diagnosis** from D6, filled in.
5. **The limitation paragraph** — what your data cannot tell you. Three sentences.

**Tonight:** re-run `Runtime > Restart and run all`, confirm `day2_clean.csv` is produced, and bring it tomorrow. Day 3 starts from that file.
